In [1]:
%env CUDA_VISIBLE_DEVICES=7

env: CUDA_VISIBLE_DEVICES=7


In [2]:
import torch
import argparse
import importlib
from typing import Dict, Any

from vace import vace_preproccess
from vace import vace_wan_inference
import torch
import argparse
import importlib
from typing import Dict, Any

from vace import vace_preproccess
from vace import vace_wan_inference
# from vace import vace_ltx_inference


def filter_args(args: Dict[str, Any], parser: argparse.ArgumentParser) -> Dict[str, Any]:
    known_args = set()
    for action in parser._actions:
        if action.dest and action.dest != "help":
            known_args.add(action.dest)
    return {k: v for k, v in args.items() if k in known_args}


# 1. 手动模拟命令行参数
argv = [
    "--base", "wan",
    "--task", "frameref",
    "--mode", "firstframe",
    "--image", "/data/gaoya/dataset/ali-vilab-VACE-Benchmark/assets/examples/firstframe/ori_image_1.png",
    "--prompt", "纪实摄影风格，前景是一位中国越野爱好者坐在越野车上，手持车载电台正在进行通联。他五官清晰，表情专注，眼神坚定地望向前方。越野车停在户外，车身略显脏污，显示出经历过的艰难路况。镜头从车外缓缓拉近，最后定格在人物的面部特写上，展现出他的坚定与热情。中景到近景，动态镜头运镜。"
]

# 2. 先只解析 --base
main_parser = argparse.ArgumentParser()
main_parser.add_argument("--base", type=str, default="ltx", choices=["ltx", "wan"])

pipeline_args, _ = main_parser.parse_known_args(argv)

if pipeline_args.base == "ltx":
    preprocess_module = vace_preproccess
    inference_module = vace_ltx_inference
else:
    preprocess_module = vace_preproccess
    inference_module = vace_wan_inference

# 3. 读取两个子 parser
preprocess_parser = preprocess_module.get_parser()
inference_parser = inference_module.get_parser()

# 4. 把子 parser 的参数并入总 parser
for parser in [preprocess_parser, inference_parser]:
    for action in parser._actions:
        if action.dest != "help":
            try:
                main_parser._add_action(action)
            except Exception:
                # 如果有重复参数，可以先跳过，或者改成 conflict_handler='resolve'
                pass

# 5. 用同一个 argv 完整解析
cli_args = main_parser.parse_args(argv)
args_dict = vars(cli_args)

print("all args:", args_dict)


pip install ltx-video@git+https://github.com/Lightricks/LTX-Video@ltx-video-0.9.1 sentencepiece --no-deps


/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/wan/modules/model.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)
/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/wan/modules/model.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)


all args: {'base': 'wan', 'task': 'frameref', 'video': None, 'image': '/data/gaoya/dataset/ali-vilab-VACE-Benchmark/assets/examples/firstframe/ori_image_1.png', 'mode': 'firstframe', 'mask': None, 'bbox': None, 'label': None, 'caption': None, 'direction': None, 'expand_ratio': None, 'expand_num': None, 'maskaug_mode': None, 'maskaug_ratio': None, 'pre_save_dir': None, 'save_fps': 16, 'model_name': 'vace-1.3B', 'size': '480p', 'frame_num': 81, 'ckpt_dir': '/data/gaoya/ckpt/Wan-AI-Wan2.1-VACE-1.3B', 'offload_model': None, 'ulysses_size': 1, 'ring_size': 1, 't5_fsdp': False, 't5_cpu': False, 'dit_fsdp': False, 'save_dir': None, 'save_file': None, 'src_video': None, 'src_mask': None, 'src_ref_images': None, 'prompt': '纪实摄影风格，前景是一位中国越野爱好者坐在越野车上，手持车载电台正在进行通联。他五官清晰，表情专注，眼神坚定地望向前方。越野车停在户外，车身略显脏污，显示出经历过的艰难路况。镜头从车外缓缓拉近，最后定格在人物的面部特写上，展现出他的坚定与热情。中景到近景，动态镜头运镜。', 'use_prompt_extend': 'plain', 'base_seed': 2025, 'sample_solver': 'unipc', 'sample_steps': None, 'sample_shift': None, 'sample_guide_scale

In [3]:

# 6. preprocess
preprocess_args = filter_args(args_dict, preprocess_parser)
preprocess_output = preprocess_module.main(preprocess_args)
print("preprocess_output:", preprocess_output)

del preprocess_module
torch.cuda.empty_cache()

# 7. inference
inference_args = filter_args(args_dict, inference_parser)
inference_args.update(preprocess_output)
inference_output = inference_module.main(inference_args)
print("inference_output:", inference_output)

Save frames result to processed/frameref/2026-03-15-07-05-31/src_video-frameref.mp4
Save frames result to processed/frameref/2026-03-15-07-05-31/src_mask-frameref.mp4
preprocess_output: {'src_video': 'processed/frameref/2026-03-15-07-05-31/src_video-frameref.mp4', 'src_mask': 'processed/frameref/2026-03-15-07-05-31/src_mask-frameref.mp4'}
[2026-03-15 07:05:31,932] INFO: offload_model is not specified, set to True.
[2026-03-15 07:05:31,933] INFO: Generation job args: Namespace(model_name='vace-1.3B', size='480p', frame_num=81, ckpt_dir='/data/gaoya/ckpt/Wan-AI-Wan2.1-VACE-1.3B', offload_model=True, ulysses_size=1, ring_size=1, t5_fsdp=False, t5_cpu=False, dit_fsdp=False, save_dir=None, save_file=None, src_video='processed/frameref/2026-03-15-07-05-31/src_video-frameref.mp4', src_mask='processed/frameref/2026-03-15-07-05-31/src_mask-frameref.mp4', src_ref_images=None, prompt='纪实摄影风格，前景是一位中国越野爱好者坐在越野车上，手持车载电台正在进行通联。他五官清晰，表情专注，眼神坚定地望向前方。越野车停在户外，车身略显脏污，显示出经历过的艰难路况。镜头从车外缓缓拉近，最后定格在人物的面部特写上，展现

100%|██████████| 50/50 [06:21<00:00,  7.62s/it]


[2026-03-15 07:13:38,840] INFO: Saving generated video to results/vace-1.3B/2026-03-15-07-13-37/out_video.mp4
[2026-03-15 07:13:39,730] INFO: Saving src_video to results/vace-1.3B/2026-03-15-07-13-37/src_video.mp4
[2026-03-15 07:13:40,651] INFO: Saving src_mask to results/vace-1.3B/2026-03-15-07-13-37/src_mask.mp4
[2026-03-15 07:13:40,654] INFO: Finished.
inference_output: {'out_video': 'results/vace-1.3B/2026-03-15-07-13-37/out_video.mp4', 'src_video': 'results/vace-1.3B/2026-03-15-07-13-37/src_video.mp4', 'src_mask': 'results/vace-1.3B/2026-03-15-07-13-37/src_mask.mp4'}
